# Phase 3C — asyncio: I/O concurrency vs CPU search

**Goal:** demonstrate when asynchronous concurrency helps (waiting for embedding/API responses) and when it does not (CPU-bound NumPy search). We do **not** call a real external API: `asyncio.sleep` simulates latency, avoiding costs/rate limits. The I/O speedups must never be represented as search-engine CPU acceleration.

**Pilot first**, then full run. Save raw timings, medians, and figures to the same Drive Phase 3 folder. No measured results until you run this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import asyncio,time,os,platform
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
OUT=ROOT/'phase3_parallelism/benchmarks';FIG=ROOT/'phase3_parallelism/figures'
OUT.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
X=np.load(EMB).astype(np.float32,copy=False)
assert X.shape==(20000,384),X.shape
X=np.ascontiguousarray(X/np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12))
print('CPU count',os.cpu_count(),'embedding shape',X.shape,'Python',platform.python_version())

In [ ]:
# Colab notebooks already have an event loop: use top-level await, not asyncio.run().
PILOT=True
N_REQUESTS=40 if PILOT else 120
LATENCY_S=0.02 if PILOT else 0.05
CONCURRENCY=[1,5,10,20] if PILOT else [1,5,10,20,40]
REPEATS=3 if PILOT else 5
SEED=20261020

async def simulated_embedding_api(request_id,latency_s):
    await asyncio.sleep(latency_s)  # simulate network/API wait, NOT embedding computation
    return request_id

async def io_serial(n,latency):
    return [await simulated_embedding_api(i,latency) for i in range(n)]

async def io_concurrent(n,latency,limit):
    semaphore=asyncio.Semaphore(limit)
    async def bounded(i):
        async with semaphore:
            return await simulated_embedding_api(i,latency)
    return await asyncio.gather(*(bounded(i) for i in range(n)))

records=[]
for c in CONCURRENCY:
    fn=io_serial if c==1 else (lambda n,latency,c=c:io_concurrent(n,latency,c))
    assert await fn(N_REQUESTS,LATENCY_S)==list(range(N_REQUESTS))
    for rep in range(REPEATS):
        t0=time.perf_counter_ns()
        result=await fn(N_REQUESTS,LATENCY_S)
        elapsed_ms=(time.perf_counter_ns()-t0)/1e6
        assert result==list(range(N_REQUESTS))
        records.append(dict(experiment='simulated_io',method='serial' if c==1 else 'asyncio',concurrency=c,n_requests=N_REQUESTS,latency_ms=LATENCY_S*1000,repeat=rep,elapsed_ms=elapsed_ms))
    print('I/O done concurrency',c)
print('Theoretical ideal serial wait:',N_REQUESTS*LATENCY_S,'seconds; actual measured separately.')

## CPU-bound negative control

The following async function **contains no await during CPU work**. Calling it with `asyncio.gather` does not magically parallelize NumPy operations; tasks run until completion on the event loop. We compare the same 20K-document top-10 retrieval against a synchronous loop. To avoid BLAS oversubscription, BLAS threads=1. Query vectors are reproducibly drawn from the corpus. Exact top-10 order should match because both variants call the same function.

In [ ]:
def topk(scores,k=10):
    ids=np.arange(len(scores))
    return np.lexsort((ids,-scores))[:k]
def cpu_search(q,corpus):
    return topk(corpus@q)
async def async_cpu_search(q,corpus):
    return cpu_search(q,corpus)  # no await: blocks the event loop

N_DOCS_CPU=5000 if PILOT else 20000
N_QUERIES_CPU=16 if PILOT else 128
corpus=X[:N_DOCS_CPU]
rng=np.random.default_rng(SEED+N_DOCS_CPU+N_QUERIES_CPU)
Q=np.ascontiguousarray(corpus[rng.choice(N_DOCS_CPU,size=N_QUERIES_CPU,replace=False)])
with threadpool_limits(limits=1,user_api='blas'):
    reference=[cpu_search(q,corpus) for q in Q]
    assert all(np.array_equal(a,b) for a,b in zip(reference,await asyncio.gather(*(async_cpu_search(q,corpus) for q in Q))))
    for rep in range(REPEATS):
        t0=time.perf_counter_ns()
        r=[cpu_search(q,corpus) for q in Q]
        records.append(dict(experiment='cpu_search',method='sequential',concurrency=1,n_requests=N_QUERIES_CPU,latency_ms=0,repeat=rep,elapsed_ms=(time.perf_counter_ns()-t0)/1e6))
        t0=time.perf_counter_ns()
        r=await asyncio.gather(*(async_cpu_search(q,corpus) for q in Q))
        records.append(dict(experiment='cpu_search',method='asyncio_gather',concurrency=N_QUERIES_CPU,n_requests=N_QUERIES_CPU,latency_ms=0,repeat=rep,elapsed_ms=(time.perf_counter_ns()-t0)/1e6))

tag='pilot' if PILOT else 'full'
df=pd.DataFrame(records)
df.to_csv(OUT/f'phase3c_asyncio_raw_{tag}.csv',index=False)
summary=df.groupby(['experiment','method','concurrency','n_requests','latency_ms'],as_index=False).agg(median_ms=('elapsed_ms','median'),mean_ms=('elapsed_ms','mean'),std_ms=('elapsed_ms','std'))
base=summary[summary.method.isin(['serial','sequential'])][['experiment','median_ms']].rename(columns={'median_ms':'baseline_ms'})
summary=summary.merge(base,on='experiment',how='left')
summary['speedup_vs_baseline']=summary.baseline_ms/summary.median_ms
summary.to_csv(OUT/f'phase3c_asyncio_summary_{tag}.csv',index=False)
print('Measured rows:',len(df))
print(summary.to_string(index=False))

In [ ]:
io=summary[summary.experiment=='simulated_io'].sort_values('concurrency')
fig,ax=plt.subplots(figsize=(8,4.5))
ax.plot(io.concurrency,io.speedup_vs_baseline,marker='o')
ax.set(xlabel='Maximum concurrent simulated requests',ylabel='Speedup vs serial',title=f'Phase 3C simulated I/O: {N_REQUESTS} requests × {LATENCY_S*1000:.0f} ms')
ax.grid(alpha=.25)
fig.tight_layout();path=FIG/f'phase3c_asyncio_io_speedup_{tag}.png';fig.savefig(path,dpi=160);plt.show()
print('Saved:',path)
cpu=summary[summary.experiment=='cpu_search']
print('CPU-only control:',cpu[['method','median_ms','speedup_vs_baseline']].to_string(index=False))

## What to report

1. **I/O concurrency**: serial vs semaphore-bounded `asyncio.gather`, 120 requests in full run. `asyncio.sleep` is a simulated wait, not a measured real API response.
2. **CPU-bound negative control**: compare identical search work with and without `asyncio.gather`; async without `await` cannot yield and does not create CPU parallelism.
3. **Limitations**: no real network, rate limits, API retries, server concurrency or embedding generation; 20K×384 CPU search is different from I/O simulation. Results are workload-specific.
4. **Design decision**: use async for I/O-bound ingestion/API calls; use batched NumPy/BLAS for in-memory similarity search. Consider combining async ingestion with CPU search only after measuring end-to-end pipeline.
5. **Instructor strategies**: NumPy/BLAS, Numba parallel (Phase 2C Gram matrix), ThreadPool, ProcessPool, asyncio. Note Numba Gram timing is **not directly comparable** to top-k search timing.
6. **Next**: send pilot results for review before full run; preserve actual CSVs and figures.